In [ ]:
#data
import pandas as pd
#plot
import matplotlib.pyplot as plt
import seaborn as sns
# interactive
from ipywidgets import interact, widgets
# sklearn
from sklearn.decomposition import PCA 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
# math
import numpy as np

import pickle

In [2]:
df_iden = pd.read_csv('data_ieee/train_identity.csv')
df_tran = pd.read_csv('data_ieee/train_transaction.csv')

In [3]:
cols_V = [c for c in df_tran.columns if any(x in c[0:2] for x in ['V'])]
cols_C = sorted([c for c in df_tran.columns if any(x in c[0:2] for x in ['C'])])

df_V = df_tran[cols_V]
df_C = df_tran[cols_C]

nan_V_counts = df_V.isna().sum().sort_values(ascending=True)

In [4]:
nan_series = df_V.isna().sum()
unique_counts = nan_series.unique()
set_V = {}
for count in sorted(unique_counts):
    cols = sorted(nan_series[nan_series == count].index.tolist())
    print(f"Count V{count}: {cols[:5]}...") 
    set_V[f'Count V{count}'] = cols
set_V[f'Count C{count}'] = cols_C

Count V12: ['V279', 'V280', 'V284', 'V285', 'V286']...
Count V314: ['V100', 'V101', 'V102', 'V103', 'V104']...
Count V1269: ['V281', 'V282', 'V283', 'V288', 'V289']...
Count V76073: ['V12', 'V13', 'V14', 'V15', 'V16']...
Count V77096: ['V53', 'V54', 'V55', 'V56', 'V57']...
Count V89164: ['V75', 'V76', 'V77', 'V78', 'V79']...
Count V168969: ['V35', 'V36', 'V37', 'V38', 'V39']...
Count V279287: ['V1', 'V10', 'V11', 'V2', 'V3']...
Count V449124: ['V220', 'V221', 'V222', 'V227', 'V234']...
Count V450721: ['V169', 'V170', 'V171', 'V174', 'V175']...
Count V450909: ['V167', 'V168', 'V172', 'V173', 'V176']...
Count V460110: ['V217', 'V218', 'V219', 'V223', 'V224']...
Count V508189: ['V322', 'V323', 'V324', 'V325', 'V326']...
Count V508589: ['V143', 'V144', 'V145', 'V150', 'V151']...
Count V508595: ['V138', 'V139', 'V140', 'V141', 'V142']...


In [6]:

def parse_string_to_list(text):
    clean_text = text.replace('[', '').replace(']', '').replace("'", "").replace('"', "")
    return [item.strip() for item in clean_text.split(',')]

parse_string_to_list(str(set_V['Count V449124']))


['V220',
 'V221',
 'V222',
 'V227',
 'V234',
 'V238',
 'V239',
 'V245',
 'V250',
 'V251',
 'V255',
 'V256',
 'V259',
 'V270',
 'V271',
 'V272']

In [8]:

results_pca = {}
pca_models = {}

for group_name, cols in set_V.items():

    df_temp = df_tran[cols].copy()
    
    df_temp = df_temp.dropna()
    
    if df_temp.empty:
        print(f"Saltando {group_name}: No quedan filas tras dropna.")
        continue
    else: 
        print(f'df_len(Nan): {len(df_V) - len(df_temp)} | {group_name}')


    pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.90))
    ])

    pipe.fit(df_temp[cols])

    
    results_pca[group_name] = {
        'original_dims': len(cols),
        'pca_dims': pipe['pca'].n_components_,
        'rows_remaining': len(df_temp)
    }
    
    pca_models[str(cols)] = pipe
    
    print(f"{group_name}: De {len(cols)} columnas a {pipe['pca'].n_components_} para 90% varianza.")


df_len(Nan): 12 | Count V12
Count V12: De 32 columnas a 8 para 90% varianza.
df_len(Nan): 314 | Count V314
Count V314: De 43 columnas a 12 para 90% varianza.
df_len(Nan): 1269 | Count V1269
Count V1269: De 11 columnas a 6 para 90% varianza.
df_len(Nan): 76073 | Count V76073
Count V76073: De 23 columnas a 8 para 90% varianza.
df_len(Nan): 77096 | Count V77096
Count V77096: De 22 columnas a 8 para 90% varianza.
df_len(Nan): 89164 | Count V89164
Count V89164: De 20 columnas a 8 para 90% varianza.
df_len(Nan): 168969 | Count V168969
Count V168969: De 18 columnas a 7 para 90% varianza.
df_len(Nan): 279287 | Count V279287
Count V279287: De 11 columnas a 6 para 90% varianza.
df_len(Nan): 449124 | Count V449124
Count V449124: De 16 columnas a 5 para 90% varianza.
df_len(Nan): 450721 | Count V450721
Count V450721: De 19 columnas a 7 para 90% varianza.
df_len(Nan): 450909 | Count V450909
Count V450909: De 31 columnas a 5 para 90% varianza.
df_len(Nan): 460110 | Count V460110
Count V460110: De 46

In [9]:

pca_results_dict = {}

def pca_transform(df_subset, pipe):
    mask = df_subset.notnull().all(axis=1)
    n_components = pipe.named_steps['pca'].n_components_
    

    results = np.full((len(df_subset), n_components), np.nan)
    
    if mask.any():
        results[mask] = pipe.transform(df_subset[mask])
    
    return results


for i, (cols_str, pipe) in enumerate(pca_models.items(), 1):
    cols = parse_string_to_list(cols_str)


    transformed_data = pca_transform(df_tran[cols], pipe)


    n_cols = transformed_data.shape[1]
    for comp_idx in range(n_cols):
        col_name = f'V_pca_{i}_cp{comp_idx + 1}'
        pca_results_dict[col_name] = transformed_data[:, comp_idx]


pca_df = pd.DataFrame(pca_results_dict, index=df_V.index)
df_V = pd.concat([df_V, pca_df], axis=1)

print(f"Added {len(pca_results_dict)} new PCA component columns.")





Added 102 new PCA component columns.


In [11]:
df_V.drop(columns=cols_V )

,V_pca_1_cp1,V_pca_1_cp2,V_pca_1_cp3,V_pca_1_cp4,V_pca_1_cp5,V_pca_1_cp6,V_pca_1_cp7,V_pca_1_cp8,V_pca_2_cp1,V_pca_2_cp2,...,V_pca_14_cp1,V_pca_14_cp2,V_pca_15_cp1,V_pca_15_cp2,V_pca_15_cp3,V_pca_15_cp4,V_pca_15_cp5,V_pca_16_cp1,V_pca_16_cp2,V_pca_16_cp3
0,-0.523342,-0.259296,-0.623897,-0.645420,0.108626,0.390720,0.138459,-0.002892,-0.434131,-0.345185,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.331272,-0.334273,-0.041043
1,-0.544028,-0.255138,-0.627948,-0.635623,0.110871,0.387395,0.139044,-0.002886,-0.461470,-0.341966,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.338282,-0.370772,-0.041321
2,-0.544028,-0.255138,-0.627948,-0.635623,0.110871,0.387395,0.139044,-0.002886,-0.461470,-0.341966,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.334589,-0.334193,-0.041046
3,1.484252,2.235579,-0.065649,-1.115356,-0.634457,-2.477608,-1.253937,-0.015963,2.136489,-0.375641,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.264718,-0.264662,-0.040293
4,-0.116997,-1.302694,0.893412,1.812249,-0.452641,0.111722,-0.410930,-0.000655,-0.461470,-0.341966,...,5.086862,-1.025491,-3.651027,0.569269,0.799549,-0.032613,-0.203059,-0.331892,-0.373663,-0.041333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,-0.496322,-0.119028,-0.610533,-0.650934,0.091763,0.232674,0.026837,-0.003266,-0.391300,-0.336972,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.320605,-0.265991,-0.040406
590536,-0.544028,-0.255138,-0.627948,-0.635623,0.110871,0.387395,0.139044,-0.002886,-0.461470,-0.341966,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.334589,-0.334193,-0.041046
590537,-0.544028,-0.255138,-0.627948,-0.635623,0.110871,0.387395,0.139044,-0.002886,-0.461470,-0.341966,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.332253,-0.310536,-0.040785
590538,0.513176,2.594172,0.424218,-0.563925,-0.261450,-1.093686,0.384814,-0.002923,0.188734,-0.244838,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.311121,-0.286562,-0.040652


In [15]:
list(pca_models.keys())

["['V279', 'V280', 'V284', 'V285', 'V286', 'V287', 'V290', 'V291', 'V292', 'V293', 'V294', 'V295', 'V297', 'V298', 'V299', 'V302', 'V303', 'V304', 'V305', 'V306', 'V307', 'V308', 'V309', 'V310', 'V311', 'V312', 'V316', 'V317', 'V318', 'V319', 'V320', 'V321']",
 "['V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V107', 'V108', 'V109', 'V110', 'V111', 'V112', 'V113', 'V114', 'V115', 'V116', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V123', 'V124', 'V125', 'V126', 'V127', 'V128', 'V129', 'V130', 'V131', 'V132', 'V133', 'V134', 'V135', 'V136', 'V137', 'V95', 'V96', 'V97', 'V98', 'V99']",
 "['V281', 'V282', 'V283', 'V288', 'V289', 'V296', 'V300', 'V301', 'V313', 'V314', 'V315']",
 "['V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34']",
 "['V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 

In [ ]:



with open('pca_models.pkl', 'wb') as f:
    pickle.dump(pca_models, f)


In [ ]:




with open('pca_models.pkl', 'rb') as f:
    pca_models = pickle.load(f)




In [21]:
pca_models

{"['V279', 'V280', 'V284', 'V285', 'V286', 'V287', 'V290', 'V291', 'V292', 'V293', 'V294', 'V295', 'V297', 'V298', 'V299', 'V302', 'V303', 'V304', 'V305', 'V306', 'V307', 'V308', 'V309', 'V310', 'V311', 'V312', 'V316', 'V317', 'V318', 'V319', 'V320', 'V321']": Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=0.9))]),
 "['V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V107', 'V108', 'V109', 'V110', 'V111', 'V112', 'V113', 'V114', 'V115', 'V116', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V123', 'V124', 'V125', 'V126', 'V127', 'V128', 'V129', 'V130', 'V131', 'V132', 'V133', 'V134', 'V135', 'V136', 'V137', 'V95', 'V96', 'V97', 'V98', 'V99']": Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=0.9))]),
 "['V281', 'V282', 'V283', 'V288', 'V289', 'V296', 'V300', 'V301', 'V313', 'V314', 'V315']": Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=0.9))]),
 "['V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V2